# Notebook 26 — Production Agent Benchmark

## The Pokémon Company — PTCG AI Battle Challenge

### Team Jesus

Notebook 25 completed the final project audit and confirmed that the submission package was ready.

Notebook 26 begins the competitive evaluation phase by benchmarking the verified production agent across repeated battle decisions.

## Purpose

This notebook evaluates the reliability, consistency, speed, and stability of the production Kaggle battle agent.

It is not a full-game simulator. Instead, it repeatedly tests the production agent against a validated battle snapshot and measures whether the same legal action is selected consistently.

## Objectives

1. Load the verified production agent from Notebook 21.
2. Confirm the production interface.
3. Create a benchmark-result data model.
4. Run repeated policy decisions.
5. Measure action consistency.
6. Measure inference latency.
7. Track runtime errors and fallbacks.
8. Validate that every returned action is legal and expected.
9. Produce scenario-level benchmark statistics.
10. Save benchmark results for future comparisons.

## Benchmark Scope

The notebook evaluates:

- decision consistency
- legal action selection
- inference latency
- runtime stability
- fallback frequency
- error frequency
- stress-test reliability

A later notebook will handle full-game simulation with evolving game states, alternating players, turns, winners, and match statistics.S
    

# Cell 2 — Imports

In [2]:
from __future__ import annotations

import importlib.util
import json
import statistics
import subprocess
import sys
import time
import types
import uuid

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

print("Python:", sys.version)
print("Working directory:", Path.cwd())

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Working directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks


# Cell 3 — Locate project files

In [4]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    markers = {
        "notebooks",
        "scripts",
        "src",
        "reports",
        "submission_work",
    }

    for candidate in [current, *current.parents]:
        found = {
            marker
            for marker in markers
            if (candidate / marker).exists()
        }

        if len(found) >= 4:
            return candidate

    return current


PROJECT_ROOT = find_project_root()

AGENT_EXPORT = (
    PROJECT_ROOT
    / "src"
    / "kaggle_agent"
    / "notebook21_export.py"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook26"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Agent export:", AGENT_EXPORT)
print("Report directory:", REPORT_DIR)

assert AGENT_EXPORT.is_file()

Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Agent export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\kaggle_agent\notebook21_export.py
Report directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook26


# Cell 4 — Load the production agent safely

In [6]:
source_21 = AGENT_EXPORT.read_text(
    encoding="utf-8-sig"
)

cleaned_lines = [
    line
    for line in source_21.splitlines()
    if line.strip() != "from __future__ import annotations"
]

cleaned_source = (
    "from __future__ import annotations\n"
    + "\n".join(cleaned_lines)
)

module_name = (
    "notebook26_agent_"
    + uuid.uuid4().hex
)

notebook21 = types.ModuleType(module_name)

notebook21.__file__ = str(AGENT_EXPORT)
notebook21.__package__ = ""

sys.modules[module_name] = notebook21

compiled_agent = compile(
    cleaned_source,
    str(AGENT_EXPORT),
    "exec",
)

exec(
    compiled_agent,
    notebook21.__dict__,
)

print("Notebook 21 production agent loaded.")

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 20 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\20_policy_engine.py
Kaggle agent package: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\kaggle_agent
Notebook 21 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook21
Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 18 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\18_kaggle_observation_adapter.py
Notebook 19 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battl

,name,module,annotations
18,ApiResult,cg.api,"[state, error]"
1,AreaType,cg.api,[]
21,Attack,cg.api,"[attackId, name, text, damage, energies]"
9,Card,cg.api,"[id, serial, playerIndex]"
20,CardData,cg.api,"[cardId, name, cardType, retreatCost, hp, weak..."
3,CardType,cg.api,[]
2,EnergyType,cg.api,[]
0,IntEnum,enum,[]
15,Log,cg.api,"[type, playerIndex, hasBasicPokemon, cardId, s..."
8,LogType,cg.api,[]


Important official classes:

- Card
- CardData
- CardType
- Log
- LogType
- Option
- OptionType
- PlayerState
- Pokemon
- SearchState
- SelectContext
- SelectData
- SelectType
- State

Card
Class: <class 'cg.api.Card'>

Annotations:
- id: <class 'int'>
- serial: <class 'int'>
- playerIndex: <class 'int'>

CardData
Class: <class 'cg.api.CardData'>

Annotations:
- cardId: <class 'int'>
- name: <class 'str'>
- cardType: <enum 'CardType'>
- retreatCost: <class 'int'>
- hp: <class 'int'>
- weakness: cg.api.EnergyType | None
- resistance: cg.api.EnergyType | None
- energyType: <enum 'EnergyType'>
- basic: <class 'bool'>
- stage1: <class 'bool'>
- stage2: <class 'bool'>
- ex: <class 'bool'>
- megaEx: <class 'bool'>
- tera: <class 'bool'>
- aceSpec: <class 'bool'>
- evolvesFrom: str | None
- skills: list[cg.api.Skill]
- attacks: list[int]

CardType
Class: <enum 'CardType'>

No annotations found.

Public attributes:
- BASIC_ENERGY
- ITEM
- POKEMON
- SPECIAL_ENERGY
- STADIUM
- SUPPORTER
- TOOL
-

,name,value
0,DECK,1
1,HAND,2
2,DISCARD,3
3,ACTIVE,4
4,BENCH,5
5,PRIZE,6
6,STADIUM,7
7,ENERGY,8
8,TOOL,9
9,PRE_EVOLUTION,10


OptionType


...

SelectContext


,name,value
0,MAIN,0
1,SETUP_ACTIVE_POKEMON,1
2,SETUP_BENCH_POKEMON,2
3,SWITCH,3
4,TO_ACTIVE,4
5,TO_BENCH,5
6,TO_FIELD,6
7,TO_HAND,7
8,DISCARD,8
9,TO_DECK,9


Official cg card objects: 1267
Notebook 17 repository: 1267

Official unique Card IDs: 1267
Repository unique Card IDs: 1267
Missing from repository: 0
Extra in repository: 0
Official card sample 1
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 1
cardType: 5
energyType: 1
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {G} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 2
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 2
cardType: 5
energyType: 2
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {R} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 3
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 3
cardType: 5
energyType: 3
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {W} Energy
resistance: None
retr

,rank,option_index,option_type,semantic_label,score,reasons
0,1,0,ATTACK,Attack with Mega Lucario ex,259.913,base=100.0 | attack_bonus=150.0 | active_energ...
1,2,1,END,End Turn,-25.000,base=0.0 | end_turn_penalty=-25.0


[OK] player_feature_extraction
[OK] battle_feature_extraction
[OK] action_feature_extraction
[OK] legal_action_ranking
[OK] best_option_index

Notebook 19 validation passed.
Notebook 19 loaded.
Production functions imported.
True
True
True
True
True

Notebook dependencies verified.
Repository size: 1267
Official CardData lookup size: 1267

Policy dependencies loaded.
BattlePolicy created.
Debug mode: True
BattlePolicy created successfully.
Safe BattlePolicy created successfully.
Option index: 0
Fallback used: True
Reason: Observation adaptation failed: AttributeError: 'NoneType' object has no attribute 'current'

Fallback behavior passed.
Synthetic snapshot loaded.
Turn: 3
Legal options: 2

Synthetic snapshot validation passed.
BattlePolicy Decision
Chosen option: 0
Fallback used: False
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex

Ranked actions:
 1. index=0   type=ATTACK     score=  259.913 Attack with Mega Lucario ex
 2. index=1   type=END        score=

# Cell 5 — Retrieve and verify production objects

In [7]:
REQUIRED_AGENT_OBJECTS = [
    "AgentStats",
    "KaggleBattleAgent",
    "kaggle_agent",
    "production_agent",
    "policy",
    "deck_ids",
    "sample_snapshot",
]

missing_objects = [
    name
    for name in REQUIRED_AGENT_OBJECTS
    if not hasattr(notebook21, name)
]

for name in REQUIRED_AGENT_OBJECTS:
    print(
        f"{'[OK]' if hasattr(notebook21, name) else '[MISSING]'} "
        f"{name}"
    )

if missing_objects:
    raise AttributeError(
        "Notebook 21 export is missing:\n"
        + "\n".join(missing_objects)
    )

KaggleBattleAgent = notebook21.KaggleBattleAgent
kaggle_agent = notebook21.kaggle_agent
production_agent = notebook21.production_agent
policy = notebook21.policy
deck_ids = notebook21.deck_ids
sample_snapshot = notebook21.sample_snapshot

assert len(deck_ids) == 60
assert callable(production_agent)
assert callable(kaggle_agent.choose)

print("\nProduction agent interface verified.")

[OK] AgentStats
[OK] KaggleBattleAgent
[OK] kaggle_agent
[OK] production_agent
[OK] policy
[OK] deck_ids
[OK] sample_snapshot

Production agent interface verified.


# Cell 6 — Define the tournament result model

In [8]:
@dataclass(frozen=True)
class TournamentRecord:
    run_number: int
    returned_action: tuple[int, ...]
    expected_action: tuple[int, ...]
    valid: bool
    elapsed_seconds: float
    agent_calls: int
    agent_errors: int
    agent_fallbacks: int

# Cell 7 — Reset agent statistics

In [9]:
kaggle_agent.calls = 0
kaggle_agent.errors = 0
kaggle_agent.fallbacks = 0

print("Agent statistics reset.")
print("Calls:", kaggle_agent.calls)
print("Errors:", kaggle_agent.errors)
print("Fallbacks:", kaggle_agent.fallbacks)

Agent statistics reset.
Calls: 0
Errors: 0
Fallbacks: 0


# Cell 8 — Run the first 100-decision tournament benchmark

In [11]:
EXPECTED_ACTION = (0,)
TOURNAMENT_RUNS = 100

tournament_records = []

for run_number in range(1, TOURNAMENT_RUNS + 1):
    started = time.perf_counter()

    returned = kaggle_agent.choose(
        sample_snapshot
    )

    elapsed = time.perf_counter() - started

    returned_action = tuple(returned)
    valid = returned_action == EXPECTED_ACTION

    tournament_records.append(
        TournamentRecord(
            run_number=run_number,
            returned_action=returned_action,
            expected_action=EXPECTED_ACTION,
            valid=valid,
            elapsed_seconds=elapsed,
            agent_calls=kaggle_agent.calls,
            agent_errors=kaggle_agent.errors,
            agent_fallbacks=kaggle_agent.fallbacks,
        )
    )

print("Tournament runs:", len(tournament_records))
print(
    "Valid decisions:",
    sum(record.valid for record in tournament_records),
)
print(
    "Invalid decisions:",
    sum(not record.valid for record in tournament_records),
)
print("Agent calls:", kaggle_agent.calls)
print("Agent errors:", kaggle_agent.errors)
print("Agent fallbacks:", kaggle_agent.fallbacks)

assert len(tournament_records) == TOURNAMENT_RUNS
assert all(record.valid for record in tournament_records)
assert kaggle_agent.errors == 0
assert kaggle_agent.fallbacks == 0

print("\nFirst tournament benchmark passed.")

Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Agent result: [0]
Reason: Selected highest-scoring legal action: Attac

# Cell 9 — Disable debug and calculate latency statistics

In [12]:
kaggle_agent.debug = False

elapsed_values = [
    record.elapsed_seconds
    for record in tournament_records
]

latency_summary = {
    "runs": len(elapsed_values),
    "minimum_seconds": min(elapsed_values),
    "maximum_seconds": max(elapsed_values),
    "average_seconds": statistics.mean(elapsed_values),
    "median_seconds": statistics.median(elapsed_values),
    "stdev_seconds": (
        statistics.stdev(elapsed_values)
        if len(elapsed_values) > 1
        else 0.0
    ),
}

for name, value in latency_summary.items():
    print(f"{name}: {value}")

assert latency_summary["runs"] == 100
assert latency_summary["average_seconds"] > 0

print("\nLatency statistics calculated.")

runs: 100
minimum_seconds: 6.609999400097877e-05
maximum_seconds: 0.00034270000469405204
average_seconds: 7.434199927956797e-05
median_seconds: 6.834999658167362e-05
stdev_seconds: 3.1579005659529774e-05

Latency statistics calculated.


# Cell 10 — Build the benchmark summary

In [13]:
valid_count = sum(
    record.valid
    for record in tournament_records
)

benchmark_summary = {
    "project": "PTCG AI Battle Challenge",
    "team": "Team Jesus",
    "scenario": "Synthetic Mega Lucario ex attack scenario",
    "runs": len(tournament_records),
    "valid_decisions": valid_count,
    "invalid_decisions": (
        len(tournament_records) - valid_count
    ),
    "accuracy": (
        valid_count / len(tournament_records)
    ),
    "agent_calls": kaggle_agent.calls,
    "agent_errors": kaggle_agent.errors,
    "agent_fallbacks": kaggle_agent.fallbacks,
    "latency": latency_summary,
}

print(json.dumps(benchmark_summary, indent=4))

assert benchmark_summary["accuracy"] == 1.0
assert benchmark_summary["agent_errors"] == 0
assert benchmark_summary["agent_fallbacks"] == 0

print("\nBenchmark summary validated.")

{
    "project": "PTCG AI Battle Challenge",
    "team": "Team Jesus",
    "scenario": "Synthetic Mega Lucario ex attack scenario",
    "runs": 100,
    "valid_decisions": 100,
    "invalid_decisions": 0,
    "accuracy": 1.0,
    "agent_calls": 100,
    "agent_errors": 0,
    "agent_fallbacks": 0,
    "latency": {
        "runs": 100,
        "minimum_seconds": 6.609999400097877e-05,
        "maximum_seconds": 0.00034270000469405204,
        "average_seconds": 7.434199927956797e-05,
        "median_seconds": 6.834999658167362e-05,
        "stdev_seconds": 3.1579005659529774e-05
    }
}

Benchmark summary validated.


# Cell 11 — Save detailed tournament results

In [14]:
TOURNAMENT_JSON = (
    REPORT_DIR
    / "tournament_benchmark.json"
)

records_payload = [
    asdict(record)
    for record in tournament_records
]

output_payload = {
    "summary": benchmark_summary,
    "records": records_payload,
}

TOURNAMENT_JSON.write_text(
    json.dumps(
        output_payload,
        indent=4,
    ),
    encoding="utf-8",
)

print("Tournament report:", TOURNAMENT_JSON)

assert TOURNAMENT_JSON.is_file()

print("\nTournament results saved.")

Tournament report: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook26\tournament_benchmark.json

Tournament results saved.


# Cell 12 — Create a leaderboard DataFrame

In [16]:
import pandas as pd

leaderboard = pd.DataFrame([
    {
        "Run": r.run_number,
        "Action": r.returned_action,
        "Correct": r.valid,
        "Latency (ms)": r.elapsed_seconds * 1000,
    }
    for r in tournament_records
])

display(leaderboard.head())

print()
print(leaderboard.describe(include="all"))

,Run,Action,Correct,Latency (ms)
0,1,"(0,)",True,0.2249
1,2,"(0,)",True,0.0959
2,3,"(0,)",True,0.0810
3,4,"(0,)",True,0.0763
4,5,"(0,)",True,0.0736



               Run Action Correct  Latency (ms)
count   100.000000    100     100    100.000000
unique         NaN      1       1           NaN
top            NaN   (0,)    True           NaN
freq           NaN    100     100           NaN
mean     50.500000    NaN     NaN      0.074342
std      29.011492    NaN     NaN      0.031579
min       1.000000    NaN     NaN      0.066100
25%      25.750000    NaN     NaN      0.067600
50%      50.500000    NaN     NaN      0.068350
75%      75.250000    NaN     NaN      0.070825
max     100.000000    NaN     NaN      0.342700


# Cell 13 — Stress Test (1000 games)

In [17]:
STRESS_RUNS = 1000

stress_results = []

for _ in range(STRESS_RUNS):
    action = kaggle_agent.choose(sample_snapshot)
    stress_results.append(tuple(action))

success_rate = (
    sum(a == (0,) for a in stress_results)
    / STRESS_RUNS
)

print("Stress Runs:", STRESS_RUNS)
print("Success Rate:", success_rate)

assert success_rate == 1.0

print("\n1000-game stress test passed.")

Stress Runs: 1000
Success Rate: 1.0

1000-game stress test passed.


# Cell 14 — Final Notebook Summary

In [18]:
print("=" * 72)
print("Notebook 26 — Tournament Benchmark")
print("=" * 72)

print()

print("Project:", benchmark_summary["project"])
print("Team:", benchmark_summary["team"])

print()

print("Tournament Runs:", benchmark_summary["runs"])
print("Accuracy:", benchmark_summary["accuracy"])
print("Errors:", benchmark_summary["agent_errors"])
print("Fallbacks:", benchmark_summary["agent_fallbacks"])

print()

print("Average Latency:",
      latency_summary["average_seconds"] * 1000,
      "ms")

print()

print("Tournament Report:",
      TOURNAMENT_JSON.name)

print()

print("NOTEBOOK 26 COMPLETED SUCCESSFULLY")

Notebook 26 — Tournament Benchmark

Project: PTCG AI Battle Challenge
Team: Team Jesus

Tournament Runs: 100
Accuracy: 1.0
Errors: 0
Fallbacks: 0

Average Latency: 0.07434199927956797 ms

Tournament Report: tournament_benchmark.json

NOTEBOOK 26 COMPLETED SUCCESSFULLY
